# Imports

In [38]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer


# Load Raw Data

In [39]:
DATA_PATH = "../data/"

train = pd.read_csv(DATA_PATH + "kaggle_b2_fraud_train_v3.csv")
test  = pd.read_csv(DATA_PATH + "kaggle_b2_fraud_test_v3.csv")


# Remove Leakage Variables

In [40]:
leakage_cols = [
    "manual_review_result",
    "post_event_status_code",
    "chargeback_resolution_time_days"
]

train.drop(columns=leakage_cols, inplace=True)
test.drop(columns=leakage_cols, inplace=True)


# Feature Engineering

### Date

In [41]:
for df in [train, test]:
    df["signup_date"] = pd.to_datetime(df["signup_date"])
    df["signup_year"] = df["signup_date"].dt.year
    df["signup_month"] = df["signup_date"].dt.month
    df["signup_dayofweek"] = df["signup_date"].dt.dayofweek
    df.drop(columns=["signup_date"], inplace=True)


# Séparer numériques / catégorielles

In [42]:
num_cols = train.select_dtypes(include=np.number).columns
cat_cols = train.select_dtypes(include="object").columns


# Pipeline de Transformation


In [43]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)


# Appliquer le pipeline

In [44]:
target = 'target_is_fraud'
id_col = 'customer_id'

In [45]:
num_cols = train.select_dtypes(include='number').columns.tolist()
num_cols.remove(target)

X_train = train[num_cols]
y_train = train[target]
X_test = test[num_cols]


In [46]:
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [47]:
X_train_processed = pipeline.fit_transform(X_train)
X_test_processed = pipeline.transform(X_test)

# Convertir en DataFrame propre

In [48]:
X_train_processed = pd.DataFrame(X_train_processed, columns=num_cols)
X_train_processed[target] = y_train.values
X_train_processed[id_col] = train[id_col].values

X_test_processed = pd.DataFrame(X_test_processed, columns=num_cols)
X_test_processed[id_col] = test[id_col].values

X_train_processed.to_csv('../data/train_preprocessed_simple.csv', index=False)
X_test_processed.to_csv('../data/test_preprocessed_simple.csv', index=False)

print('Export terminé.')


Export terminé.
